In [1]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os
import joblib

In [2]:
# -----------------------------
# SETTINGS
# -----------------------------
MAX_BACKGROUND = 20       # Background samples for SHAP
MAX_EXPLAIN = 10          # Test samples for SHAP
MAX_FEATURES_FOR_SHAP = 100  # Reduce features to this number
TOP_K = 20                # Top features to show in plots
RANDOM_STATE = 42
OUT_DIR = "./shap_best_tcga_results"
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
# -----------------------------
# LOAD PIPELINE
# -----------------------------
pipeline = joblib.load("svm.pkl")
svm_model = pipeline.named_steps['svm']
print(f"SVM kernel: {svm_model.kernel}")

SVM kernel: poly


c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
X_train = pd.read_csv('../Datasets/X_train.csv')
X_test  = pd.read_csv('../Datasets/X_test.csv')

In [5]:
# -----------------------------
# PREPARE DATA
# -----------------------------
X_train_arr = X_train.values if hasattr(X_train, "values") else X_train
X_test_arr = X_test.values if hasattr(X_test, "values") else X_test

In [6]:
# -----------------------------
# FEATURE REDUCTION
# -----------------------------
n_features_total = X_train_arr.shape[1]

if n_features_total > MAX_FEATURES_FOR_SHAP:
    print(f"Reducing features from {n_features_total} to {MAX_FEATURES_FOR_SHAP}")

    # Select top features by variance
    feature_vars = np.var(X_train_arr, axis=0)
    top_indices = np.argsort(feature_vars)[-MAX_FEATURES_FOR_SHAP:]
    top_indices = sorted(top_indices)

    X_train_reduced = X_train_arr[:, top_indices]
    X_test_reduced = X_test_arr[:, top_indices]

    feature_names = X_train.columns[top_indices].tolist() if hasattr(X_train, "columns") else [f"Feature_{i}" for i in top_indices]
else:
    top_indices = np.arange(n_features_total)
    X_train_reduced = X_train_arr
    X_test_reduced = X_test_arr
    feature_names = X_train.columns.tolist() if hasattr(X_train, "columns") else [f"Feature_{i}" for i in range(n_features_total)]

print(f"Using {X_train_reduced.shape[1]} features for SHAP")

Reducing features from 19277 to 100
Using 100 features for SHAP


In [7]:
# -----------------------------
# SAMPLE BACKGROUND + TEST
# -----------------------------
def shap_sample(X, n, random_state):
    idx = np.random.RandomState(random_state).choice(len(X), size=n, replace=False)
    return X[idx]

background_reduced = shap_sample(X_train_reduced, min(MAX_BACKGROUND, len(X_train_reduced)), RANDOM_STATE)
X_explain_reduced = shap_sample(X_test_reduced, min(MAX_EXPLAIN, len(X_test_reduced)), RANDOM_STATE)

print("Background shape:", background_reduced.shape)
print("Explain shape:", X_explain_reduced.shape)

Background shape: (20, 100)
Explain shape: (10, 100)


In [8]:
# -----------------------------
# MODEL WRAPPER FOR REDUCED FEATURES
# -----------------------------
def model_wrapper(X_reduced):
    """
    Expands X_reduced (selected features) to full feature space
    Non-selected features set to zero
    """
    batch_size = X_reduced.shape[0]
    X_full = np.zeros((batch_size, X_train_arr.shape[1]))
    X_full[:, top_indices] = X_reduced
    return pipeline.predict_proba(X_full)

In [9]:
# -----------------------------
# INITIALIZE SHAP EXPLAINER
# -----------------------------
print("\nInitializing SHAP Explainer...")
explainer = shap.Explainer(model_wrapper, background_reduced)

# Compute SHAP values
shap_values = explainer(X_explain_reduced)
gc.collect()
print("SHAP values computed successfully")


Initializing SHAP Explainer...


c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
PermutationExplainer explainer:  10%|█         | 1/10 [00:00<?, ?it/s]c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
c:\Users\Alvin\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
PermutationExplainer

SHAP values computed successfully


In [10]:
# -----------------------------
# PROCESS SHAP VALUES
# -----------------------------
if isinstance(shap_values.values, list):
    shap_arr = np.stack([np.array(s) for s in shap_values.values], axis=2)  # (samples, features, classes)
else:
    shap_arr = np.array(shap_values.values)

print(f"SHAP values shape: {shap_arr.shape}")

SHAP values shape: (10, 100, 5)


In [11]:
# -----------------------------
# OVERALL FEATURE IMPORTANCE
# -----------------------------
if shap_arr.ndim == 3:
    shap_overall = np.abs(shap_arr).max(axis=2)
else:
    shap_overall = np.abs(shap_arr)

mean_abs_shap = shap_overall.mean(axis=0)

df_shap = pd.DataFrame({
    "Feature": feature_names,
    "MeanAbsSHAP": mean_abs_shap
}).sort_values(by="MeanAbsSHAP", ascending=False)

df_shap.head(TOP_K).to_csv(os.path.join(OUT_DIR, "top_features_overall.csv"), index=False)

# Plot top features
plt.figure(figsize=(10, 8))
sns.barplot(data=df_shap.head(TOP_K), x="MeanAbsSHAP", y="Feature", palette="viridis")
plt.title(f"Top {TOP_K} Features by Mean |SHAP|")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "top_features_bar.png"), dpi=150)
plt.close()

C:\Users\Alvin\AppData\Local\Temp\ipykernel_27496\1859592440.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_shap.head(TOP_K), x="MeanAbsSHAP", y="Feature", palette="viridis")


In [12]:
# -----------------------------
# PER-CLASS SUMMARY PLOTS (if multi-class)
# -----------------------------
if shap_arr.ndim == 3:
    n_classes = shap_arr.shape[2]
    classes = pipeline.named_steps['svm'].classes_
    for class_idx in range(n_classes):
        class_label = classes[class_idx]
        class_shap = shap_arr[:, :, class_idx]
        plt.figure(figsize=(12, 10))
        shap.summary_plot(class_shap, X_explain_reduced, feature_names=feature_names, max_display=TOP_K, show=False)
        plt.title(f"SHAP Summary - {class_label}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"shap_summary_{class_label}.png"), dpi=150)
        plt.close()

print(f"\nSHAP analysis complete. Results saved to: {OUT_DIR}")

C:\Users\Alvin\AppData\Local\Temp\ipykernel_27496\3814199151.py:11: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(class_shap, X_explain_reduced, feature_names=feature_names, max_display=TOP_K, show=False)
C:\Users\Alvin\AppData\Local\Temp\ipykernel_27496\3814199151.py:11: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(class_shap, X_explain_reduced, feature_names=feature_names, max_display=TOP_K, show=False)
C:\Users\Alvin\AppData\Local\Temp\ipykernel_27496\3814199151.py:11: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer u


SHAP analysis complete. Results saved to: ./shap_best_tcga_results
